# 💻 Laboratorio ETL: Monitoreo de Riesgo Cambiario (USD/CLP)
Este Notebook documenta el paso a paso de nuestro pipeline de datos. Cruzamos el valor del dólar en tiempo real (API Mindicador) con nuestro inventario en bodegas (MySQL) y nuestras reglas de negocio (CSV) para alertar sobre quiebres de margen de ganancia.

In [1]:
import pandas as pd
import requests
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


# 💻 Laboratorio ETL: Monitoreo de Riesgo Cambiario (USD/CLP)
## FASE 0: Inicialización del Entorno (One-Click Setup)
Las siguientes celdas configuran la base de datos MySQL, construyen las tablas necesarias y ejecutan el pipeline ETL automatizado para poblar el Data Warehouse. **(Ejecutar solo 1 vez al iniciar el proyecto).**

In [2]:
# 1. Instalar dependencias necesarias (Descomentar si es la primera vez en este PC)
!pip install pandas requests sqlalchemy pymysql dash plotly

print("Verificación de librerías completada.")

Verificación de librerías completada.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 2. Construir la Base de Datos y las Tablas en MySQL
!python etl/setup_mysql.py

2026-06-21 01:46:50,239 - [INFO] - Creando base de datos retail_fx_db...
2026-06-21 01:46:50,292 - [INFO] - ¡Tablas MySQL creadas exitosamente para el Retail!


In [4]:
# 3. Ejecutar el Motor ETL (Extracción API, Cruce de Datos y Carga)
!python etl/etl_riesgo_fx.py

     fecha        producto     region_cl  stock_unidades  costo_usd  valor_dolar  costo_total_clp  precio_venta_clp  margen_actual_pct    estado_riesgo
2026-05-07  MacBook Pro M3 Metropolitana              45     1500.0       892.83       1339245.00           1500000          10.717000 OK (Margen Sano)
2026-05-07  MacBook Pro M3        Biobío              15     1500.0       892.83       1339245.00           1550000          13.597097 OK (Margen Sano)
2026-05-07 Monitor Dell 4K Metropolitana              30      400.0       892.83        357132.00            450000          20.637333 OK (Margen Sano)
2026-05-07   iPhone 15 Pro    Valparaíso             120      999.0       892.83        891937.17           1050000          15.053603 OK (Margen Sano)
2026-05-08  MacBook Pro M3 Metropolitana              45     1500.0       887.71       1331565.00           1500000          11.229000 OK (Margen Sano)
2026-05-08  MacBook Pro M3        Biobío              15     1500.0       887.71       1

2026-06-21 01:46:53,587 - [INFO] - 1. EXTRACT: Iniciando extracción de las 3 fuentes...
2026-06-21 01:46:54,672 - [INFO] - Extraídos 31 días de cotización del dólar.
2026-06-21 01:46:54,790 - [INFO] - Extraído el inventario actual de MySQL.
2026-06-21 01:46:54,804 - [INFO] - Extraídas reglas de negocio del CSV.
2026-06-21 01:46:54,804 - [INFO] - 2. TRANSFORM: Cruzando datos y calculando riesgo cambiario...
2026-06-21 01:46:54,848 - [INFO] - 3. LOAD: Guardando tabla analítica en MySQL...
2026-06-21 01:46:54,932 - [INFO] - ¡Pipeline ETL completado con éxito! Listo para el Dashboard.


### FASE 1: Extracción (Extract)
Consumimos la API pública del gobierno para obtener el valor del dólar de los últimos 31 días.

In [5]:
# Extracción API Mindicador
url = "https://mindicador.cl/api/dolar"
respuesta = requests.get(url)
df_api = pd.DataFrame(respuesta.json()['serie'])
df_api['fecha'] = pd.to_datetime(df_api['fecha']).dt.date
df_api = df_api.rename(columns={'valor': 'valor_dolar'})

print("Muestra de datos de la API (Dólar):")
display(df_api.head(3))

Muestra de datos de la API (Dólar):


,fecha,valor_dolar
0,2026-06-19,897.19
1,2026-06-18,883.46
2,2026-06-17,886.53


### FASE 2: Integración de Orígenes Internos
Extraemos el inventario actual desde nuestro ERP (MySQL) y las reglas de márgenes mínimos permitidos (CSV).

In [6]:
# Conexión a MySQL
engine = create_engine('mysql+pymysql://root:@localhost/retail_fx_db')
df_mysql = pd.read_sql("SELECT * FROM bodegas_tech", engine)

# Lectura del CSV
df_csv = pd.read_csv("data/reglas_negocio.csv")

print("Inventario en MySQL (Costo en USD):")
display(df_mysql)

print("\nReglas de Negocio CSV (Precio de Venta en CLP y Margen Mínimo):")
display(df_csv)

Inventario en MySQL (Costo en USD):


,id_producto,producto,stock_unidades,region_cl,costo_usd
0,SKU-MAC,MacBook Pro M3,45,Metropolitana,1500.0
1,SKU-IPH,iPhone 15 Pro,120,Valparaíso,999.0
2,SKU-MON,Monitor Dell 4K,30,Metropolitana,400.0
3,SKU-MAC,MacBook Pro M3,15,Biobío,1500.0



Reglas de Negocio CSV (Precio de Venta en CLP y Margen Mínimo):


,producto,region_cl,precio_venta_clp,margen_minimo_pct
0,MacBook Pro M3,Metropolitana,1500000,10.0
1,iPhone 15 Pro,Valparaíso,1050000,12.0
2,Monitor Dell 4K,Metropolitana,450000,15.0
3,MacBook Pro M3,Biobío,1550000,10.0


### FASE 3: Transformación 
Unimos todas las fuentes y aplicamos la matemática financiera. Calculamos el costo real en pesos chilenos y el porcentaje de margen de ganancia para disparar las alertas rojas.

In [7]:
# 1. Cruzamos Inventario con Reglas
df_inventario = pd.merge(df_mysql, df_csv, on=['producto', 'region_cl'], how='inner')

# 2. Cross Join con los días del Dólar
df_master = df_inventario.merge(df_api, how='cross')

# 3. Matemática de Negocio
df_master['costo_total_clp'] = df_master['costo_usd'] * df_master['valor_dolar']
df_master['margen_actual_pct'] = ((df_master['precio_venta_clp'] - df_master['costo_total_clp']) / df_master['precio_venta_clp']) * 100

# 4. Regla de Alerta
df_master['estado_riesgo'] = df_master.apply(
    lambda row: 'ALERTA ROJA' if row['margen_actual_pct'] < row['margen_minimo_pct'] else 'OK (Sano)', 
    axis=1
)

df_final = df_master[['fecha', 'producto', 'region_cl', 'valor_dolar', 'costo_total_clp', 'margen_actual_pct', 'estado_riesgo']]

print("Data Warehouse Final Procesado (Cruce Maestro):")
display(df_final.sort_values(by='fecha').head(10))

Data Warehouse Final Procesado (Cruce Maestro):


,fecha,producto,region_cl,valor_dolar,costo_total_clp,margen_actual_pct,estado_riesgo
61,2026-05-07,iPhone 15 Pro,Valparaíso,892.83,891937.17,15.053603,OK (Sano)
92,2026-05-07,Monitor Dell 4K,Metropolitana,892.83,357132.00,20.637333,OK (Sano)
30,2026-05-07,MacBook Pro M3,Metropolitana,892.83,1339245.00,10.717000,OK (Sano)
123,2026-05-07,MacBook Pro M3,Biobío,892.83,1339245.00,13.597097,OK (Sano)
60,2026-05-08,iPhone 15 Pro,Valparaíso,887.71,886822.29,15.540734,OK (Sano)
29,2026-05-08,MacBook Pro M3,Metropolitana,887.71,1331565.00,11.229000,OK (Sano)
91,2026-05-08,Monitor Dell 4K,Metropolitana,887.71,355084.00,21.092444,OK (Sano)
122,2026-05-08,MacBook Pro M3,Biobío,887.71,1331565.00,14.092581,OK (Sano)
121,2026-05-11,MacBook Pro M3,Biobío,890.89,1336335.00,13.784839,OK (Sano)
28,2026-05-11,MacBook Pro M3,Metropolitana,890.89,1336335.00,10.911000,OK (Sano)


### FASE 4: Carga (Load)
Una vez validado el modelo, los datos se cargan limpios en la base de datos MySQL (`dw_riesgo_fx`) para ser consumidos en tiempo real por el Dashboard en Dash/Plotly.

In [8]:
# df_final.to_sql('dw_riesgo_fx', engine, if_exists='replace', index=False)
print("Carga a MySQL simulada con éxito.")

Carga a MySQL simulada con éxito.
